In [1]:
import numpy as np

In [2]:
import pandas as pd

In [3]:
df = pd.DataFrame({"category": ["a", "a", "a", "a", "b", "b", "b", "b"], "data": np.random.standard_normal(8), "weights": np.random.uniform(size=8)})

In [4]:
df

,category,data,weights
0,a,-1.118416,0.771661
1,a,-0.135300,0.564831
2,a,-0.462921,0.480855
3,a,0.352427,0.003064
4,b,0.478816,0.715080
5,b,1.750749,0.208574
6,b,-0.537092,0.849453
7,b,0.876310,0.677557


In [5]:
grouped = df.groupby("category")

In [8]:
def get_wavg(group):
    return np.average(group["data"], weights= group["weights"])

In [9]:
grouped.apply(get_wavg)

category
a   -0.637756
b    0.344833
dtype: float64

In [10]:
close_px = pd.read_csv("examples/stock_px.csv", parse_dates=True, index_col=0)

In [11]:
close_px.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 2214 entries, 2003-01-02 to 2011-10-14
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2214 non-null   float64
 1   MSFT    2214 non-null   float64
 2   XOM     2214 non-null   float64
 3   SPX     2214 non-null   float64
dtypes: float64(4)
memory usage: 86.5 KB


In [12]:
close_px.tail(4)

,AAPL,MSFT,XOM,SPX
2011-10-11,400.29,27.00,76.27,1195.54
2011-10-12,402.19,26.96,77.16,1207.25
2011-10-13,408.43,27.18,76.37,1203.66
2011-10-14,422.00,27.27,78.11,1224.58


In [19]:
def spx_corr(group):
    return group.corrwith(group["SPX"])

In [20]:
rets = close_px.pct_change().dropna()

In [21]:
def get_year(x):
    return x.year

In [22]:
by_year = rets.groupby(get_year)

In [23]:
by_year.apply(spx_corr)

,AAPL,MSFT,XOM,SPX
2003,0.541124,0.745174,0.661265,1.0
2004,0.374283,0.588531,0.557742,1.0
2005,0.467540,0.562374,0.631010,1.0
2006,0.428267,0.406126,0.518514,1.0
2007,0.508118,0.658770,0.786264,1.0
2008,0.681434,0.804626,0.828303,1.0
2009,0.707103,0.654902,0.797921,1.0
2010,0.710105,0.730118,0.839057,1.0
2011,0.691931,0.800996,0.859975,1.0


In [24]:
def corr_aapl_msft(group):
    return group["AAPL"].corr(group["MSFT"])

In [25]:
by_year.apply(corr_aapl_msft)

2003    0.480868
2004    0.259024
2005    0.300093
2006    0.161735
2007    0.417738
2008    0.611901
2009    0.432738
2010    0.571946
2011    0.581987
dtype: float64

In [26]:
import statsmodels.api as sm
def regress(data, yvar = None, xvars = None):
    Y = data[yvar]
    X = data[xvars]
    X["intercept"] = 1
    result = sm.OLS(Y, X).fit()
    return result.params

In [27]:
by_year.apply(regress, yvar = "AAPL", xvars = ["SPX"])

,SPX,intercept
2003,1.195406,0.000710
2004,1.363463,0.004201
2005,1.766415,0.003246
2006,1.645496,0.000080
2007,1.198761,0.003438
2008,0.968016,-0.001110
2009,0.879103,0.002954
2010,1.052608,0.001261
2011,0.806605,0.001514
